# 06 · Plate grammar decoding

The YOLO26s reader finds every character of a plate as its own box. Today the boxes above 0.25 confidence are simply
sorted left to right, so a plate can come out with seven digits (the dash read as ১), a letter twice, or no letter.

Every Bangladeshi plate follows one pattern: **district, optional word (Metro / Chatto / Raj), one series letter,
six digits**. The grammar decoder keeps the reader's low-confidence guesses as well and picks the plate that follows
this pattern:

- characters that overlap are one position; its class guesses are the alternatives,
- exactly six digits (the six most confident), one district, one letter, at most one word,
- a missing district or letter is recovered from a weaker guess (down to 0.05 confidence).

In [ ]:
%pip install -q ultralytics==8.4.46

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

import pandas as pd
import torch
from ultralytics import YOLO

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
DATA = next(p for p in INPUT.rglob("bangla-plate-reading") if p.is_dir())
WEIGHTS = sorted(INPUT.rglob("best.pt"))
assert len(WEIGHTS) == 1, f"attach exactly one reader best.pt, found {WEIGHTS}"
WEIGHTS = WEIGHTS[0]

CLASSES = json.loads((DATA / "classes.json").read_text(encoding="utf-8"))
BANGLA = [c["bangla"] for c in CLASSES]
KIND = [c["kind"] for c in CLASSES]

IMGSZ = 320
KEEP = 0.25      # confidence the AI service uses today
RECOVER = 0.05   # weakest guess the decoder may use to fill a missing district or letter
DEVICE = 0 if torch.cuda.is_available() else "cpu"

model = YOLO(WEIGHTS)
images = sorted((DATA / "images" / "test").iterdir())
print(f"data {DATA}\nweights {WEIGHTS}\n{len(images)} test crops on {DEVICE}")

## Plate text and error measures (same for every method)

In [ ]:
def plate_text(boxes):
    """boxes: (class, x_center). District, word and letter left to right, then the digits left to right."""
    boxes = sorted(boxes, key=lambda b: b[1])
    head = " ".join(BANGLA[c] for c, _ in boxes if KIND[c] != "digit")
    digits = "".join(BANGLA[c] for c, _ in boxes if KIND[c] == "digit")
    if len(digits) == 6:
        digits = f"{digits[:2]}-{digits[2:]}"
    return " ".join(part for part in (head, digits) if part)


def edit_distance(a, b):
    row = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, row[0] = row[0], i
        for j, cb in enumerate(b, 1):
            prev, row[j] = row[j], min(row[j] + 1, row[j - 1] + 1, prev + (ca != cb))
    return row[-1]

## Reader guesses

One pass at confidence 0.05 with per-class NMS, so a character can keep several class guesses. The baseline is the subset at 0.25: per-class NMS never lets a weak box remove a stronger one, so this equals a run at 0.25.

In [ ]:
candidates = {}
for start in range(0, len(images), 64):
    batch = images[start:start + 64]
    for path, result in zip(batch, model.predict(batch, imgsz=IMGSZ, conf=RECOVER, device=DEVICE,
                                                 max_det=100, verbose=False)):
        b = result.boxes
        candidates[path.name] = [
            {"cls": int(c), "conf": float(p), "x": float(xywh[0]), "box": [float(v) for v in xyxy]}
            for c, p, xywh, xyxy in zip(b.cls.tolist(), b.conf.tolist(), b.xywhn.tolist(), b.xyxyn.tolist())
        ]
print(sum(len(v) for v in candidates.values()), "guesses on", len(candidates), "crops")

## Grammar decoder

In [ ]:
NEED = {"digit": 6, "district": 1, "letter": 1, "word": 1}


def iou(a, b):
    w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = w * h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union else 0.0


def positions(cands):
    """Overlapping guesses are one character position; its guesses are the alternatives, best first."""
    groups = []
    for c in sorted(cands, key=lambda c: -c["conf"]):
        for g in groups:
            if iou(g[0]["box"], c["box"]) > 0.5:
                g.append(c)
                break
        else:
            groups.append([c])
    return groups


def grammar_decode(cands, recover=RECOVER):
    groups = positions(cands)
    chosen, used = [], set()

    # every position votes with its best guess; keep the most confident ones up to the plate pattern
    by_kind = defaultdict(list)
    for i, g in enumerate(groups):
        if g[0]["conf"] >= KEEP:
            by_kind[KIND[g[0]["cls"]]].append((g[0]["conf"], i, g[0]))
    for kind, n in NEED.items():
        for conf, i, guess in sorted(by_kind[kind], key=lambda t: -t[0])[:n]:
            chosen.append(guess)
            used.add(i)

    # a plate always has a district, a letter and six digits: fill gaps from weaker guesses
    for kind in ("district", "letter", "digit"):
        missing = NEED[kind] - sum(KIND[c["cls"]] == kind for c in chosen)
        options = []
        for i, g in enumerate(groups):
            if i in used:
                continue
            alt = [c for c in g if KIND[c["cls"]] == kind and c["conf"] >= recover]
            if alt:
                options.append((alt[0]["conf"], i, alt[0]))
        for conf, i, guess in sorted(options, key=lambda t: -t[0])[:max(0, missing)]:
            chosen.append(guess)
            used.add(i)

    return [(c["cls"], c["x"]) for c in chosen]

## Score

- **baseline**: today's AI service (guesses ≥ 0.25, left to right)
- **grammar**: the plate pattern only, no weak guesses (recovery off)
- **grammar + recovery**: the full decoder

In [ ]:
METHODS = {
    "baseline": lambda cands: [(c["cls"], c["x"]) for c in cands if c["conf"] >= KEEP],
    "grammar": lambda cands: grammar_decode(cands, recover=KEEP),
    "grammar + recovery": lambda cands: grammar_decode(cands),
}

rows = []
for path in images:
    label = DATA / "labels" / "test" / f"{path.stem}.txt"
    truth = [(int(p[0]), float(p[1])) for p in (l.split() for l in label.read_text().splitlines()) if p]
    t = plate_text(truth)
    for method, decode in METHODS.items():
        p = plate_text(decode(candidates[path.name]))
        rows.append({"image": path.name, "source": path.name.split("__")[0], "method": method,
                     "truth": t, "predicted": p, "correct": t == p,
                     "errors": edit_distance(t, p), "length": len(t)})

scores = pd.DataFrame(rows)
scores.to_csv(WORK / "grammar_decoding_test_plates.csv", index=False)


def summary(df):
    return pd.Series({"plates": len(df), "plate acc %": 100 * df["correct"].mean(),
                      "CER %": 100 * df["errors"].sum() / df["length"].sum()})


table = scores.groupby(["source", "method"], sort=False).apply(summary).round(2)
overall = scores.groupby("method", sort=False).apply(summary).round(2)
overall.index = pd.MultiIndex.from_product([["all"], overall.index])
print(pd.concat([table, overall]).to_string())